In [ ]:
# Install Unsloth: It makes fine-tuning 2x faster and uses 70% less memory.
!pip install unsloth
# Install supporting libraries for handling the model and data.
!pip install --no-deps xformers trl peft accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 123.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.4/418.4 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 118.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 125.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 110.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

In [ ]:
from unsloth import FastLanguageModel
import torch

# Load the base Llama-3 model in 4-bit to save GPU memory.
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name = "unsloth/llama-3-8b-bnb-4bit",
#     max_seq_length = 2048,
#     load_in_4bit = True,
# )
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-0.6B-bnb-4bit",
    max_seq_length = 2048,
    dtype = None,          # auto
    load_in_4bit = True    # saves memory
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.4: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/539M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

unsloth/Qwen3-0.6B-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [ ]:

# Add 'LoRA' adapters: Instead of training the whole model,
# Standard LoRA setup as before
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
)

Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.4.4 patched 28 layers with 28 QKV layers, 28 O layers and 0 MLP layers.


In [ ]:
import json
from datasets import Dataset

# Load your uploaded file.
with open('companies_dataset.json', 'r') as f:
    raw_data = json.load(f)

# This template tells the model how to read your data.
prompt_template = """Instruction: Provide information about the company.
Company Name: {}
Response: {}"""

def format_data(example):
    # We combine the company details into a single response block.
    response_text = f"URL: {example['URL']}, Category: {example['Category']}, Description: {example['Description']}"
    return { "text" : prompt_template.format(example['Name'], response_text) }

# Convert JSON to a format ready for training.
dataset = Dataset.from_list(raw_data).map(format_data)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    args = TrainingArguments(
        per_device_train_batch_size = 2, # Number of rows processed at once.
        gradient_accumulation_steps = 4, # Simulates a larger batch for stability.
        max_steps = 500,                # How many "lessons" to run.
        learning_rate = 2e-4,           # The speed of learning.
        fp16 = True,                    # Uses faster math (Half-precision).
        logging_steps = 1,               # Show loss for every single step.
        output_dir = "outputs",
    ),
)

trainer.train() # This starts the actual fine-tuning.

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/500 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 8 | Total steps = 500
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 4,587,520 of 600,637,440 (0.76% trained)


Step,Training Loss
1,1.297907
2,1.420814
3,1.069909
4,1.504162
5,1.335862
6,1.311298
7,1.491264
8,1.412323
9,1.576178
10,1.438277


Step,Training Loss
1,1.297907
2,1.420814
3,1.069909
4,1.504162
5,1.335862
6,1.311298
7,1.491264
8,1.412323
9,1.576178
10,1.438277


TrainOutput(global_step=500, training_loss=1.0633877876996993, metrics={'train_runtime': 523.4652, 'train_samples_per_second': 7.641, 'train_steps_per_second': 0.955, 'total_flos': 1269902066319360.0, 'train_loss': 1.0633877876996993, 'epoch': 7.944})

In [ ]:
# Save locally first
model.save_pretrained("saas_model_lora")
tokenizer.save_pretrained("saas_model_lora")

# Export to GGUF (Quantized to q8_0 for high quality)
model.save_pretrained_gguf("model_gguf", tokenizer, quantization_method = "q8_0")

Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:00<00:00, 7133.17it/s]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:06<00:00,  6.58s/it]


Unsloth: Merge process complete. Saved to `/content/model_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q8_0'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['model_gguf_gguf/Qwen3-0.6B.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q8_0. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['model_gguf_gguf/Qwen3-0.6B.Q8_0.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cl

{'save_directory': 'model_gguf',
 'gguf_directory': 'model_gguf_gguf',
 'gguf_files': ['model_gguf_gguf/Qwen3-0.6B.Q8_0.gguf'],
 'modelfile_location': 'model_gguf_gguf/Modelfile',
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}

In [ ]:
# Cell 10: Zip and download the LoRA adapter
!zip -r saas_model_lora.zip saas_model_lora/

# Download to your computer
from google.colab import files
files.download("saas_model_lora.zip")
files.download("model_gguf_gguf/Qwen3-0.6B.Q8_0.gguf")
files.download("model_gguf_gguf/Modelfile")
print("✅ Download started! Check your browser downloads.")

updating: saas_model_lora/ (stored 0%)
updating: saas_model_lora/README.md (deflated 65%)
updating: saas_model_lora/chat_template.jinja (deflated 77%)
updating: saas_model_lora/adapter_config.json (deflated 57%)
updating: saas_model_lora/tokenizer_config.json (deflated 43%)
updating: saas_model_lora/adapter_model.safetensors (deflated 7%)
updating: saas_model_lora/tokenizer.json (deflated 81%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download started! Check your browser downloads.
